# SkinCare AI — Clasificación multimodal de lesiones cutáneas

Este notebook documenta el entrenamiento y la exportación de un clasificador
binario basado en **EfficientNetV2-B0**. El modelo combina una imagen
dermatoscópica con tres variables de contexto: edad aproximada, sexo codificado
y localización anatómica.

> **Aviso de seguridad:** este proyecto es educativo y experimental. No es un
> dispositivo médico, no ofrece un diagnóstico y no sustituye la evaluación de
> un profesional sanitario.

## Objetivo y alcance

El flujo cubre:

1. lectura de TFRecords y preprocesamiento multimodal;
2. entrenamiento inicial a 256 × 256 píxeles;
3. fine-tuning parcial de EfficientNetV2-B0;
4. evaluación con métricas adecuadas para clases desbalanceadas;
5. un experimento independiente de *progressive resizing* a 384 × 384;
6. exportación del mejor checkpoint de 256 × 256 a TensorFlow Lite.

Kaggle es el entorno de referencia. Las versiones exactas de los datasets, sus
licencias y las limitaciones del modelo se documentan en
[`DATASETS.md`](../docs/DATASETS.md) y
[`MODEL_CARD.md`](../docs/MODEL_CARD.md).

## 1. Entorno y reproducibilidad

Se importan las dependencias, se registra el entorno de ejecución y se fijan
semillas para Python, NumPy y TensorFlow. Una misma semilla reduce variaciones,
pero no garantiza resultados idénticos entre versiones de CUDA, cuDNN,
TensorFlow o hardware distinto.

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os
import gc
import json
import warnings
import platform
import random
import cv2
from tqdm import tqdm
import zipfile
import shutil

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (confusion_matrix, roc_auc_score, roc_curve, 
                           matthews_corrcoef, balanced_accuracy_score,
                           precision_recall_curve, average_precision_score,
                           f1_score, recall_score, precision_score)
from sklearn.utils import class_weight

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import (ModelCheckpoint, EarlyStopping, 
                                       ReduceLROnPlateau, TensorBoard, Callback)
import tensorflow.keras.backend as K

warnings.filterwarnings('ignore')

print(f"Python: {platform.python_version()}")
print(f"TensorFlow: {tf.__version__}")
print(f"NumPy: {np.__version__}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU configurada: {gpus}")
    except RuntimeError as e:
        print(e)
else:
    print("No se detectó ninguna GPU, usando CPU")

AUTO = tf.data.experimental.AUTOTUNE

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 2. Lectura, normalización y aumento de datos

Cada TFRecord contiene la imagen, la etiqueta binaria y tres metadatos. Las
imágenes JPEG se convierten a `float32` y se escalan al intervalo `[0, 1]`.
Los metadatos se transforman en un vector de tres componentes.

El aumento de datos se aplica únicamente a la imagen: reflexiones, pequeños
cambios de tono, saturación, contraste y brillo. El vector de metadatos no se
modifica.

In [ ]:
def decode_image(image_data):
    # Decodifica el JPEG empaquetado
    image = tf.image.decode_jpeg(image_data, channels=3)
    # Convierte a float32 y normaliza de 0-255 a 0-1
    image = tf.cast(image, tf.float32) / 255.0
    # Asegura que TensorFlow conozca el tamaño exacto usando la variable global IMAGE_SIZE
    image = tf.reshape(image, [*IMAGE_SIZE, 3])
    return image

def read_labeled_tfrecord(example):
    LABELED_TFREC_FORMAT = {
        "image": tf.io.FixedLenFeature([], tf.string),
        "target": tf.io.FixedLenFeature([], tf.int64),
        "age_approx": tf.io.FixedLenFeature([], tf.int64, default_value=-1),
        "sex": tf.io.FixedLenFeature([], tf.int64, default_value=-1),
        "anatom_site_general_challenge": tf.io.FixedLenFeature([], tf.int64, default_value=-1),
    }
    example = tf.io.parse_single_example(example, LABELED_TFREC_FORMAT)
    
    image = decode_image(example['image'])
    
    # Preprocesamiento de metadatos (Normalización básica)
    age = tf.cast(example['age_approx'], tf.float32) / 100.0  
    sex = tf.cast(example['sex'], tf.float32)                 
    site = tf.cast(example['anatom_site_general_challenge'], tf.float32) / 7.0 
    
    metadata = tf.stack([age, sex, site])
    target = tf.cast(example['target'], tf.float32)
    
    return (image, metadata), target

def data_augment(inputs, target):
    # 1. Desempaquetamos las entradas
    image, metadata = inputs
    
    # 2. Hacemos la aumentación SOLO a la imagen
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_hue(image, max_delta=0.01)
    image = tf.image.random_saturation(image, lower=0.7, upper=1.3)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    image = tf.image.random_brightness(image, max_delta=0.1)
    
    # 3. Volvemos a empaquetar todo exactamente como el modelo lo espera
    return (image, metadata), target

## 3. Datasets y partición a 256 × 256

El entrenamiento usa TFRecords de SIIM-ISIC 2020 e ISIC 2019 adjuntos al
notebook de Kaggle. Los shards de 2020 se ordenan y se dividen 80/20; todos los
shards de 2019 se añaden al entrenamiento.

Esta es una partición por shards. Antes de interpretar las métricas como una
estimación definitiva de generalización debe verificarse explícitamente que no
exista solapamiento de pacientes entre entrenamiento y validación.

In [ ]:
IMAGE_SIZE = [256, 256]
BATCH_SIZE = 64

# Rutas a los datasets en Kaggle (asegúrate de haber añadido ambos)
GCS_PATH_2020 = '/kaggle/input/datasets/cdeotte/melanoma-256x256'
GCS_PATH_2019 = '/kaggle/input/datasets/cdeotte/isic2019-256x256'

# Obtenemos las listas de todos los archivos .tfrec
FILES_2020 = tf.io.gfile.glob(GCS_PATH_2020 + '/train*.tfrec')
FILES_2019 = tf.io.gfile.glob(GCS_PATH_2019 + '/train*.tfrec')

if not FILES_2020 or not FILES_2019:
    raise FileNotFoundError(
        'No se encontraron los TFRecords de 256 px. '
        'Adjunta los datasets indicados en docs/DATASETS.md.'
    )

# Ordenarlos para asegurar reproducibilidad
FILES_2020.sort()
FILES_2019.sort()

# Separar el 20% de 2020 para validación
split_index = int(len(FILES_2020) * 0.8)
VALIDATION_FILENAMES = FILES_2020[split_index:]

# El entrenamiento son los restantes de 2020 + TODOS los de 2019
TRAINING_FILENAMES = FILES_2020[:split_index] + FILES_2019

print(f"Archivos TFRecord para Entrenamiento: {len(TRAINING_FILENAMES)}")
print(f"Archivos TFRecord para Validación: {len(VALIDATION_FILENAMES)}")

def get_training_dataset():
    dataset = tf.data.TFRecordDataset(TRAINING_FILENAMES, num_parallel_reads=AUTO)
    dataset = dataset.map(read_labeled_tfrecord, num_parallel_calls=AUTO)
    dataset = dataset.map(data_augment, num_parallel_calls=AUTO)
    dataset = dataset.repeat() # Infinito para las épocas
    dataset = dataset.shuffle(2048) # Mezclador en memoria
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTO) # Precarga
    return dataset

def get_validation_dataset():
    dataset = tf.data.TFRecordDataset(VALIDATION_FILENAMES, num_parallel_reads=AUTO)
    dataset = dataset.map(read_labeled_tfrecord, num_parallel_calls=AUTO)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.cache() # Guarda en RAM para validar rapidísimo
    dataset = dataset.prefetch(AUTO)
    return dataset

# --- INSTANCIAMOS LOS DATASETS ---
train_generator = get_training_dataset()
val_generator = get_validation_dataset()

# Calculamos los pasos por época
NUM_TRAINING_IMAGES = len(TRAINING_FILENAMES) * 2071
NUM_VALIDATION_IMAGES = len(VALIDATION_FILENAMES) * 2071
STEPS_PER_EPOCH = NUM_TRAINING_IMAGES // BATCH_SIZE

print(f"Imágenes de entrenamiento aproximadas: {NUM_TRAINING_IMAGES:,}")
print(f"Imágenes de validación aproximadas: {NUM_VALIDATION_IMAGES:,}")
print(f"Pasos por época calculados: {STEPS_PER_EPOCH}")

## 4. Función de pérdida y métricas

El melanoma es la clase minoritaria, por lo que la exactitud global puede ser
engañosa. Se utiliza *focal loss* para aumentar el peso de los ejemplos
difíciles y se registran ROC AUC, PR AUC, precisión y sensibilidad.

La selección del mejor checkpoint se basa en `val_pr_auc`, una métrica más
informativa que `accuracy` cuando las clases están fuertemente desbalanceadas.

In [ ]:
def focal_loss(gamma=2.0, alpha=0.75):
    def focal_loss_fixed(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        
        y_true = tf.cast(y_true, tf.float32)
        
        p_t = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        alpha_factor = tf.where(tf.equal(y_true, 1), alpha, 1 - alpha)
        
        cross_entropy = -tf.math.log(p_t)
        weight = alpha_factor * tf.pow((1 - p_t), gamma)
        
        loss = weight * cross_entropy
        return tf.reduce_mean(loss)
    return focal_loss_fixed

class F1Score(tf.keras.metrics.Metric):
    def __init__(self, name='f1_score', **kwargs):
        super(F1Score, self).__init__(name=name, **kwargs)
        self.precision = tf.keras.metrics.Precision()
        self.recall = tf.keras.metrics.Recall()
        
    def update_state(self, y_true, y_pred, sample_weight=None):
        self.precision.update_state(y_true, y_pred, sample_weight)
        self.recall.update_state(y_true, y_pred, sample_weight)
        
    def result(self):
        precision = self.precision.result()
        recall = self.recall.result()
        return 2 * ((precision * recall) / (precision + recall + tf.keras.backend.epsilon()))
    
    def reset_state(self):
        self.precision.reset_state()
        self.recall.reset_state()

## 5. Arquitectura multimodal

La rama visual utiliza EfficientNetV2-B0 preentrenada con ImageNet. Sus
características se combinan con una pequeña rama densa para los metadatos.
Después de la fusión, una capa sigmoide produce una puntuación binaria.

**Contrato de entrada:**

- imagen RGB `float32`: `[batch, 256, 256, 3]`, escalada a `[0, 1]`;
- metadatos `float32`: `[batch, 3]`;
- salida sigmoide: `[batch, 1]`.

In [ ]:
def build_multimodal_model(img_size=256, num_metadata=3):
    # --- Rama 1: Procesamiento de Imagen ---
    input_img = layers.Input(shape=(img_size, img_size, 3), name='input_img')
    base_model = tf.keras.applications.EfficientNetV2B0(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights='imagenet'
    )
    x_img = base_model(input_img)
    x_img = layers.GlobalAveragePooling2D()(x_img)
    
    # --- Rama 2: Procesamiento de Metadatos ---
    input_meta = layers.Input(shape=(num_metadata,), name='input_meta')
    x_meta = layers.Dense(16, activation='relu')(input_meta)
    x_meta = layers.BatchNormalization()(x_meta)
    
    # --- Fusión de ambas ramas ---
    concat = layers.Concatenate()([x_img, x_meta])
    
    # Capas densas finales para la decisión
    x = layers.Dense(128, activation='relu')(concat)
    x = layers.Dropout(0.3)(x)
    output = layers.Dense(1, activation='sigmoid', name='output')(x)
    
    model = models.Model(inputs=[input_img, input_meta], outputs=output)
    return model, base_model

model, base_model = build_multimodal_model(img_size=256)

print(f"Parámetros totales: {model.count_params():,}")
print(f"Parámetros entrenables: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")

## 6. Fase 1 — Entrenamiento inicial

La primera fase entrena el modelo con Adam y un *learning rate* con
`CosineDecay`. Se guarda el checkpoint con mejor `val_pr_auc` y se aplica
*early stopping* para evitar continuar cuando la validación deja de mejorar.

In [ ]:
EPOCHS_FASE_1 = 5

initial_learning_rate = 0.001
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate,
    decay_steps=STEPS_PER_EPOCH * EPOCHS_FASE_1,
    alpha=0.01
)

optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule, clipnorm=1.0)

model.compile(
    optimizer=optimizer,
    loss=focal_loss(gamma=2.0, alpha=0.75), 
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc', curve='ROC'),
        tf.keras.metrics.AUC(name='pr_auc', curve='PR'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

print("Modelo compilado y listo para TFRecords")

os.makedirs('checkpoints', exist_ok=True)

callbacks = [
    ModelCheckpoint(
        'checkpoints/best_model_efficientnet.weights.h5',
        monitor='val_pr_auc', 
        mode='max',
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_pr_auc',
        patience=3,
        restore_best_weights=True,
        mode='max',
        verbose=1
    )
]

print("\n" + "="*60)
print("FASE 1: ENTRENAMIENTO INICIAL (TFRECORDS)")
print("="*60)

history_phase1 = model.fit(
    train_generator,
    validation_data=val_generator,
    steps_per_epoch=STEPS_PER_EPOCH,
    epochs=EPOCHS_FASE_1,
    callbacks=callbacks,
    verbose=1
)

## 7. Fase 2 — Fine-tuning

Se habilita el backbone y se mantienen congeladas sus capas iniciales. Las
últimas 50 capas se ajustan con un *learning rate* menor para adaptar las
representaciones visuales sin destruir de forma brusca los pesos preentrenados.

El mejor modelo completo de esta fase se guarda como
`checkpoints/best_model_multimodal.keras`. Este es el checkpoint utilizado por
la exportación móvil al final del notebook.

In [ ]:
print("\n" + "="*60)
print("FASE 2: FINE-TUNING (Descongelando capas)")
print("="*60)

# Descongelamos el modelo base
base_model.trainable = True

# Congelamos las primeras capas para no destruir lo que EfficientNet ya sabe de formas básicas
fine_tune_at = len(base_model.layers) - 50
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Recompilamos con un learning rate reducido para el fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5, clipnorm=1.0),
    loss=focal_loss(gamma=2.0, alpha=0.75),
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc', curve='ROC'),
        tf.keras.metrics.AUC(name='pr_auc', curve='PR'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

print(f"Capas entrenables ahora: {len([l for l in model.layers if l.trainable])}")

# Entrenamos hasta siete épocas adicionales
EPOCHS_FASE_2 = 7

callbacks_fase2 = [
    # Guardamos en el nuevo formato seguro .keras
    ModelCheckpoint(
        'checkpoints/best_model_multimodal.keras', 
        monitor='val_pr_auc', 
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    
    # Detenemos el entrenamiento cuando deja de mejorar val_pr_auc
    EarlyStopping(
        monitor='val_pr_auc',
        patience=4,
        restore_best_weights=True,
        mode='max',
        verbose=1
    )
]

history_phase2 = model.fit(
    train_generator,
    validation_data=val_generator,
    steps_per_epoch=STEPS_PER_EPOCH,
    epochs=EPOCHS_FASE_1 + EPOCHS_FASE_2,
    initial_epoch=EPOCHS_FASE_1,
    callbacks=callbacks_fase2,
    verbose=1
)

## 8. Evaluación del modelo de 256 × 256

El umbral de decisión se selecciona sobre la curva ROC buscando una
sensibilidad mínima del 85 %. Después se calculan ROC AUC, average precision,
sensibilidad, especificidad, precisión, NPV, F1, MCC y balanced accuracy.

El resultado registrado de referencia está en
[`MODEL_CARD.md`](../docs/MODEL_CARD.md). Es una evaluación interna sobre el
split de validación, no una validación clínica ni externa.

In [ ]:
from sklearn.metrics import roc_curve
import numpy as np

print("Extrayendo predicciones del set de Validación...")

y_true = []
y_pred = []

# Iteramos sobre el dataset de validación que está cacheado en RAM
for (batch_images, batch_meta), batch_labels in val_generator:
    # Le pasamos la lista de ambas entradas al modelo
    preds = model.predict([batch_images, batch_meta], verbose=0)
    y_pred.extend(preds.flatten())
    y_true.extend(batch_labels.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Encontramos el umbral óptimo
def find_optimal_threshold_arrays(y_true, y_pred, min_sensitivity=0.85):
    fpr, tpr, thresholds = roc_curve(y_true, y_pred)
    valid_idx = np.where(tpr >= min_sensitivity)[0]
    
    if len(valid_idx) > 0:
        optimal_idx = valid_idx[0]
    else:
        optimal_idx = np.argmax(tpr - fpr)
        
    optimal_threshold = thresholds[optimal_idx]
    return optimal_threshold, fpr[optimal_idx], tpr[optimal_idx]

optimal_threshold, fpr_opt, tpr_opt = find_optimal_threshold_arrays(y_true, y_pred, min_sensitivity=0.85)

print(f"\nUmbral óptimo calculado: {optimal_threshold:.4f}")
print(f"Sensibilidad garantizada: {tpr_opt:.2%}")
print(f"Especificidad lograda: {1-fpr_opt:.2%}")

def evaluate_model_comprehensive(y_true, y_pred, threshold=0.5, suffix=""):
    y_pred_binary = (y_pred >= threshold).astype(int)

    cm = confusion_matrix(y_true, y_pred_binary)
    tn, fp, fn, tp = cm.ravel()

    metrics = {
        'threshold': threshold,
        'auc_roc': roc_auc_score(y_true, y_pred),
        'average_precision': average_precision_score(y_true, y_pred),
        'sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0,
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'precision': tp / (tp + fp) if (tp + fp) > 0 else 0,
        'npv': tn / (tn + fn) if (tn + fn) > 0 else 0,
        'f1_score': f1_score(y_true, y_pred_binary),
        'mcc': matthews_corrcoef(y_true, y_pred_binary),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred_binary),
        'nnd': 1 / (tp / (tp + fn)) if tp > 0 else float('inf'),
        'confusion_matrix': cm
    }
    
    # Imprimir resultados
    print("\n" + "="*60)
    print("EVALUACIÓN DEL MODELO")
    print("="*60)
    print(f"Umbral: {threshold:.4f}")
    print(f"\nMétricas de Rendimiento:")
    print(f"  - AUC-ROC: {metrics['auc_roc']:.4f}")
    print(f"  - Average Precision: {metrics['average_precision']:.4f}")
    print(f"  - Sensibilidad (Recall): {metrics['sensitivity']:.2%}")
    print(f"  - Especificidad: {metrics['specificity']:.2%}")
    print(f"  - Precisión: {metrics['precision']:.2%}")
    print(f"  - NPV: {metrics['npv']:.2%}")
    print(f"  - F1-Score: {metrics['f1_score']:.4f}")
    print(f"  - MCC: {metrics['mcc']:.4f}")
    print(f"  - Balanced Accuracy: {metrics['balanced_accuracy']:.2%}")
    print(f"  - NND: {metrics['nnd']:.2f}")
    return metrics

test_metrics = evaluate_model_comprehensive(y_true, y_pred, optimal_threshold)

## 9. Experimento de *progressive resizing* a 384 × 384

Esta sección crea un pipeline independiente con imágenes de mayor resolución y
un batch menor. Se libera memoria antes de construir el nuevo modelo.

La fase de 384 × 384 es experimental: parte del checkpoint de pesos de la fase
inicial y **no** es el modelo exportado a TensorFlow Lite por este notebook.
Mantener esta frontera explícita evita atribuir sus resultados al artefacto
móvil de 256 × 256.

In [ ]:
print("Limpiando memoria para Progressive Resizing...")
del train_generator
del val_generator
gc.collect()
K.clear_session()

IMAGE_SIZE = [384, 384]
BATCH_SIZE = 16

GCS_PATH_2020_384 = '/kaggle/input/datasets/cdeotte/melanoma-384x384'
GCS_PATH_2019_384 = '/kaggle/input/datasets/cdeotte/isic2019-384x384'

FILES_2020_384 = tf.io.gfile.glob(GCS_PATH_2020_384 + '/train*.tfrec')
FILES_2019_384 = tf.io.gfile.glob(GCS_PATH_2019_384 + '/train*.tfrec')

if not FILES_2020_384 or not FILES_2019_384:
    raise FileNotFoundError(
        'No se encontraron los TFRecords de 384 px. '
        'Adjunta los datasets indicados en docs/DATASETS.md.'
    )

FILES_2020_384.sort()
FILES_2019_384.sort()

split_index_384 = int(len(FILES_2020_384) * 0.8)
VALIDATION_FILENAMES_384 = FILES_2020_384[split_index_384:]
TRAINING_FILENAMES_384 = FILES_2020_384[:split_index_384] + FILES_2019_384

def get_training_dataset_384():
    dataset = tf.data.TFRecordDataset(TRAINING_FILENAMES_384, num_parallel_reads=AUTO)
    dataset = dataset.map(read_labeled_tfrecord, num_parallel_calls=AUTO)
    dataset = dataset.map(data_augment, num_parallel_calls=AUTO)
    dataset = dataset.repeat()
    dataset = dataset.shuffle(512)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTO)
    return dataset

def get_validation_dataset_384():
    dataset = tf.data.TFRecordDataset(VALIDATION_FILENAMES_384, num_parallel_reads=AUTO)
    dataset = dataset.map(read_labeled_tfrecord, num_parallel_calls=AUTO)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTO)
    return dataset

train_generator_384 = get_training_dataset_384()
val_generator_384 = get_validation_dataset_384()

NUM_TRAINING_IMAGES_384 = len(TRAINING_FILENAMES_384) * 2071
NUM_VALIDATION_IMAGES_384 = len(VALIDATION_FILENAMES_384) * 2071
STEPS_PER_EPOCH_384 = NUM_TRAINING_IMAGES_384 // BATCH_SIZE

print(f"Pasos por época (384x384): {STEPS_PER_EPOCH_384}")

## 10. Entrenamiento experimental a 384 × 384

Se construye una nueva instancia de la arquitectura, se cargan los pesos
compatibles de la fase inicial y se ajustan las últimas capas con un learning
rate reducido. Su checkpoint se guarda de forma separada.

In [ ]:
print("\n" + "="*60)
print("FASE 3: PROGRESSIVE RESIZING (384x384)")
print("="*60)

model_384, base_model_384 = build_multimodal_model(img_size=384)
model_384.load_weights('checkpoints/best_model_efficientnet.weights.h5')

base_model_384.trainable = True
fine_tune_at_384 = len(base_model_384.layers) - 50
for layer in base_model_384.layers[:fine_tune_at_384]:
    layer.trainable = False

model_384.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6, clipnorm=1.0),
    loss=focal_loss(gamma=2.0, alpha=0.75),
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc', curve='ROC'),
        tf.keras.metrics.AUC(name='pr_auc', curve='PR'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

EPOCHS_FASE_3 = 2

callbacks_phase3 = [
    ModelCheckpoint(
        'checkpoints/best_model_efficientnet_384.weights.h5',
        monitor='val_pr_auc', 
        mode='max',
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
]

history_phase3 = model_384.fit(
    train_generator_384,
    validation_data=val_generator_384,
    steps_per_epoch=STEPS_PER_EPOCH_384,
    epochs=EPOCHS_FASE_3,
    callbacks=callbacks_phase3,
    verbose=1
)

## 11. Evaluación experimental a 384 × 384

La evaluación repite el mismo criterio de sensibilidad mínima y el mismo
conjunto de métricas. Estos resultados deben compararse con los de 256 × 256
solo después de verificar que ambos pipelines usan exactamente el mismo split
y contrato de metadatos.

In [ ]:
from sklearn.metrics import roc_curve
import numpy as np

print("Extrayendo predicciones del set de Validación (384x384 Multimodal)...")

# Reiniciamos los acumuladores para la evaluación de 384 px
y_true = []
y_pred = []

# Recorremos el generador multimodal de validación
# Desempaquetamos las imágenes y los metadatos
for (batch_images, batch_meta), batch_labels in val_generator_384:
    
    # Ejecutamos ambas entradas del modelo
    preds = model_384.predict([batch_images, batch_meta], verbose=0)
    
    # Acumulamos probabilidades y etiquetas
    y_pred.extend(preds.flatten())
    y_true.extend(batch_labels.numpy())

# Convertimos los acumuladores a NumPy
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Encontramos el umbral óptimo
def find_optimal_threshold_arrays(y_true, y_pred, min_sensitivity=0.85):
    fpr, tpr, thresholds = roc_curve(y_true, y_pred)
    valid_idx = np.where(tpr >= min_sensitivity)[0]
    
    if len(valid_idx) > 0:
        optimal_idx = valid_idx[0]
    else:
        optimal_idx = np.argmax(tpr - fpr)
        
    optimal_threshold = thresholds[optimal_idx]
    return optimal_threshold, fpr[optimal_idx], tpr[optimal_idx]

optimal_threshold_384, fpr_opt, tpr_opt = find_optimal_threshold_arrays(y_true, y_pred, min_sensitivity=0.85)

print(f"\nUmbral óptimo calculado final: {optimal_threshold_384:.4f}")

# Aplicamos la misma evaluación utilizada en la resolución de 256 px
test_metrics = evaluate_model_comprehensive(y_true, y_pred, optimal_threshold_384)

## 12. Exportación del modelo de 256 × 256

La exportación carga `best_model_multimodal.keras`, el mejor checkpoint completo
de la fase 2, y genera `skincare_multimodal_256.tflite` con optimización
dinámica de pesos.

Este artefacto requiere dos entradas y no es intercambiable con el modelo
single-input de 224 × 224 que actualmente consume la aplicación Android. El
procedimiento de paridad necesario antes de una integración se describe en
[`ANDROID_INTEGRATION.md`](../docs/ANDROID_INTEGRATION.md).

In [ ]:
import tensorflow as tf
import os

print("\n" + "="*60)
print("SECCIÓN 12: EXPORTACIÓN A TFLITE (MODELO MULTIMODAL 256x256)")
print("="*60)

model_name_keras = 'skincare_model_multimodal.keras'
model.save(model_name_keras)
print(f"✓ Modelo guardado en formato Keras: {model_name_keras}")

# 1. Cargamos el modelo ganador de la Fase 2
print("Cargando el mejor modelo de la Fase 2...")
best_model_path = 'checkpoints/best_model_multimodal.keras' # O .keras si lo cambiaste
final_model = tf.keras.models.load_model(best_model_path, compile=False)

# 2. Convertimos el modelo a formato TensorFlow Lite
print("Iniciando conversión a TFLite...")
converter = tf.lite.TFLiteConverter.from_keras_model(final_model)

# Opcional pero MUY RECOMENDADO para móviles: Optimización de peso
# Esto comprime el modelo de ~25MB a unos ~7MB sin perder apenas precisión
converter.optimizations = [tf.lite.Optimize.DEFAULT]

try:
    tflite_model = converter.convert()
    
    # 3. Guardamos el archivo
    tflite_filename = 'skincare_multimodal_256.tflite'
    with open(tflite_filename, 'wb') as f:
        f.write(tflite_model)
        
    # Calculamos el peso en MB
    model_size_mb = os.path.getsize(tflite_filename) / (1024 * 1024)
    print(f"\n¡ÉXITO! Modelo exportado a: {tflite_filename}")
    print(f"Peso final para la app móvil: {model_size_mb:.2f} MB")

except Exception as e:
    print(f"Error durante la conversión: {e}")

## Artefactos y trazabilidad

Una ejecución completa debe conservar fuera de Git:

- el checkpoint `.keras`;
- la exportación `.tflite`;
- el identificador de versión del notebook de Kaggle;
- las versiones exactas del entorno y de los datasets;
- los hashes SHA-256 de los artefactos;
- las métricas y el umbral obtenidos en esa misma ejecución.

Los binarios aprobados se publican como *release assets* y se registran en
[`artifacts/manifest.json`](../artifacts/manifest.json). El repositorio no
incluye imágenes médicas ni credenciales de Kaggle.